<a href="https://colab.research.google.com/github/irfanalam05/Machine-Learning/blob/main/Eightfold_Round1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!mkdir eightfold_submission

In [2]:
!mkdir eightfold_submission

mkdir: cannot create directory ‘eightfold_submission’: File exists


In [3]:
%%writefile eightfold_submission/agent.py

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils import compute_sample_weight


def _fit_model(X, y):
    if len(np.unique(y)) < 2:
        clf = LogisticRegression(max_iter=500, class_weight="balanced")
        clf.fit(X, y)
        return clf

    sw = compute_sample_weight("balanced", y)
    clf = GradientBoostingClassifier(
        n_estimators=150,
        learning_rate=0.08,
        max_depth=3,
        random_state=42
    )
    clf.fit(X, y, sample_weight=sw)
    return clf


def run_agent(df: pd.DataFrame, oracle_fn, budget: int) -> np.ndarray:

    n = len(df)
    scaler = StandardScaler()
    X = scaler.fit_transform(df.values)

    # -------- STAGE 1: Diversity Sampling --------
    from sklearn.cluster import KMeans
    k = min(30, budget // 3)
    kmeans = KMeans(n_clusters=k, random_state=42)
    clusters = kmeans.fit_predict(X)

    selected = []
    for i in range(k):
        cluster_points = np.where(clusters == i)[0]
        if len(cluster_points) > 0:
            selected.append(cluster_points[0])

    selected = list(set(selected))[:budget]

    labels = oracle_fn(selected)

    # -------- STAGE 2: Exploitation --------
    remaining = budget - len(selected)

    if remaining > 0 and len(np.unique(labels)) > 1:
        model = _fit_model(X[selected], np.array(labels))
        probs = model.predict_proba(X)[:, 1]

        not_selected = np.array([i for i in range(n) if i not in set(selected)])
        top_indices = not_selected[np.argsort(-probs[not_selected])[:remaining]]

        more_labels = oracle_fn(top_indices.tolist())

        selected.extend(top_indices.tolist())
        labels.extend(more_labels)

    # -------- FINAL TRAIN --------
    final_model = _fit_model(X[selected], np.array(labels))
    predictions = final_model.predict(X)

    return predictions.astype(int)

Writing eightfold_submission/agent.py


In [4]:
%%writefile eightfold_submission/manifest.json

{
  "team_name": "Quantum Crew",
  "team_id": "mohdirfanalam77886@gmail.com",
  "institution": "Galgotias University",
  "members": [
    {
      "name": "MOHD IRFAN ALAM",
      "email": "mohdirfanalam77886@gmail.com",
      "role": "Leader,Active Learning Strategy & Model Design"
    },
    {
      "name": "Khushboo Kumari",
      "email": "khushwork02@gmail.com",
      "role": "Member,Model Analysis & Validation"
    }
  ],
  "entry_point": "agent.py"
}

Writing eightfold_submission/manifest.json


In [5]:
%%writefile eightfold_submission/package.sh
#!/bin/bash
echo "No extra dependencies required"

Writing eightfold_submission/package.sh


In [6]:
!zip -r submission.zip eightfold_submission

  adding: eightfold_submission/ (stored 0%)
  adding: eightfold_submission/manifest.json (deflated 43%)
  adding: eightfold_submission/package.sh (stored 0%)
  adding: eightfold_submission/agent.py (deflated 59%)
